# Experiment 9 - Gradient Descent Variants on MNIST

**Aim:** Compare six gradient-based optimizers on the same dense MNIST
classifier and measure how optimizer choice affects test accuracy and
convergence.

**Theory in brief:** Gradient descent minimizes a loss by moving parameters in
the direction of the negative gradient. The variants differ in how they use
one sample, a mini-batch, past gradients, or per-parameter learning rates:

- **SGD:** one sample per update; noisy but frequent.
- **Mini-batch SGD:** averages over 32 samples for a more stable step.
- **Momentum:** adds a fraction of the previous update to smooth the path.
- **Nesterov:** evaluates the gradient at the anticipated future position.
- **Adagrad:** divides each parameter's rate by accumulated squared gradients.
- **Adam:** combines momentum with adaptive second-moment scaling.

**Dataset:** MNIST, downloaded at runtime from
`https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz`.
No local dataset is used.

**Expected result (original lab record):**

| Optimizer | Test accuracy |
|---|---|
| SGD | 97.42% |
| Mini-Batch SGD | 94.46% |
| Momentum | 97.34% |
| Nesterov | 97.23% |
| Adagrad | 90.50% |
| Adam | 97.05% |

**Runtime:** the full protocol is 6 models x 5 epochs. It is fastest on a GPU,
but the batch-size-1 SGD run is still the slowest member. Set
`FAST_DEV_RUN = True` for a small smoke test.

## 1. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## 2. Download and split MNIST

The lab protocol uses 48,000 training images, 12,000 validation images, and
the standard 10,000-image test set. `FAST_DEV_RUN` reduces the sizes and epochs
for a quick end-to-end check without changing the model.

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"


def load_mnist():
    path = tf.keras.utils.get_file(
        fname="mnist.npz",
        origin=MNIST_URL,
        cache_subdir="datasets",
    )
    with np.load(path) as data:
        return (
            (np.array(data["x_train"]), np.array(data["y_train"])),
            (np.array(data["x_test"]), np.array(data["y_test"])),
        )


(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = load_mnist()

X_train_full = (X_train_raw.astype("float32") / 255.0).reshape(-1, 784)
X_test_full = (X_test_raw.astype("float32") / 255.0).reshape(-1, 784)

print(f"Full training data: {X_train_full.shape}")
print(f"Full test data    : {X_test_full.shape}")
print(f"Pixel range       : [{X_train_full.min():.2f}, {X_train_full.max():.2f}]")

assert X_train_full.shape == (60000, 784)
assert X_test_full.shape == (10000, 784)

In [ ]:
FAST_DEV_RUN = False

if FAST_DEV_RUN:
    TRAIN_SAMPLES = 12000
    VAL_SAMPLES = 3000
    TEST_SAMPLES = 2000
    EPOCHS = 2
else:
    TRAIN_SAMPLES = 48000
    VAL_SAMPLES = 12000
    TEST_SAMPLES = 10000
    EPOCHS = 5

Xtr = X_train_full[:TRAIN_SAMPLES]
ytr = y_train_raw[:TRAIN_SAMPLES]
Xval = X_train_full[TRAIN_SAMPLES:TRAIN_SAMPLES + VAL_SAMPLES]
yval = y_train_raw[TRAIN_SAMPLES:TRAIN_SAMPLES + VAL_SAMPLES]
Xte = X_test_full[:TEST_SAMPLES]
yte = y_test_raw[:TEST_SAMPLES]

val_dataset = tf.data.Dataset.from_tensor_slices((Xval, yval)).batch(256).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((Xte, yte)).batch(256).prefetch(tf.data.AUTOTUNE)

print(f"Fast dev run : {FAST_DEV_RUN}")
print(f"Epochs       : {EPOCHS}")
print(f"Train shape  : {Xtr.shape}")
print(f"Val shape    : {Xval.shape}")
print(f"Test shape   : {Xte.shape}")
print("Train labels :", np.bincount(ytr).tolist())

## 3. Model and optimizer configurations

Every optimizer trains a fresh copy of the same 784 -> 128 ReLU -> 64 ReLU ->
10 softmax network. The random seed is reset before each model so the
comparison starts from identical weights.

In [ ]:
def build_dense_mnist_model(optimizer, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(784,), name="flattened_digit"),
            tf.keras.layers.Dense(128, activation="relu", name="hidden_1"),
            tf.keras.layers.Dense(64, activation="relu", name="hidden_2"),
            tf.keras.layers.Dense(10, activation="softmax", name="digit_probabilities"),
        ],
        name="dense_mnist_classifier",
    )
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


OPTIMIZER_SPECS = {
    "SGD": (lambda: tf.keras.optimizers.SGD(), 1),
    "MiniBatch": (lambda: tf.keras.optimizers.SGD(), 32),
    "Momentum": (lambda: tf.keras.optimizers.SGD(0.01, momentum=0.9), 32),
    "Nesterov": (lambda: tf.keras.optimizers.SGD(0.01, momentum=0.9, nesterov=True), 32),
    "Adagrad": (lambda: tf.keras.optimizers.Adagrad(), 32),
    "Adam": (lambda: tf.keras.optimizers.Adam(), 32),
}

build_dense_mnist_model(OPTIMIZER_SPECS["Adam"][0]()).summary()

## 4. Train all six optimizers

In [ ]:
histories = {}
trained_models = {}
results = []

for name, (optimizer_factory, batch_size) in OPTIMIZER_SPECS.items():
    print(f"\n===== {name} (batch size {batch_size}) =====")
    model = build_dense_mnist_model(optimizer_factory())
    start_time = time.perf_counter()
    history = model.fit(
        Xtr,
        ytr,
        batch_size=batch_size,
        epochs=EPOCHS,
        validation_data=val_dataset,
        verbose=2,
    )
    elapsed = time.perf_counter() - start_time

    test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)

    histories[name] = history
    trained_models[name] = model
    results.append(
        {
            "Optimizer": name,
            "Batch size": batch_size,
            "Test loss": float(test_loss),
            "Test accuracy": float(test_accuracy),
            "Seconds": float(elapsed),
        }
    )

results_df = pd.DataFrame(results)
print()
print(results_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
results_df.to_csv(OUTPUT_DIR / "exp09_optimizer_results.csv", index=False)

## 5. Validation curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for name in OPTIMIZER_SPECS:
    epochs_axis = np.arange(1, len(histories[name].history["val_loss"]) + 1)
    axes[0].plot(epochs_axis, histories[name].history["val_loss"], marker="o", label=name)
    axes[1].plot(epochs_axis, histories[name].history["val_accuracy"], marker="o", label=name)

axes[0].set_title("Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Sparse categorical cross-entropy")
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.0)
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp09_validation_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Test accuracy and training time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]

bars = axes[0].bar(results_df["Optimizer"], results_df["Test accuracy"], color=colors)
axes[0].set_ylim(0.80, 1.005)
axes[0].set_ylabel("Test accuracy")
axes[0].set_title("Test accuracy by optimizer")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y", alpha=0.3)
for bar, value in zip(bars, results_df["Test accuracy"]):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{value:.4f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

axes[1].bar(results_df["Optimizer"], results_df["Seconds"], color=colors)
axes[1].set_ylabel("Training seconds")
axes[1].set_title("Training time by optimizer")
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp09_accuracy_and_time.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Confusion matrix and report for the best model

In [ ]:
best_index = int(results_df["Test accuracy"].idxmax())
best_row = results_df.loc[best_index]
best_model = trained_models[best_row["Optimizer"]]

print(f"Best optimizer: {best_row['Optimizer']}")
print(f"Test accuracy : {best_row['Test accuracy']:.4f}")
print(f"Test loss     : {best_row['Test loss']:.4f}")

best_probabilities = best_model.predict(Xte, verbose=0)
best_predictions = best_probabilities.argmax(axis=1)

print()
print(
    classification_report(
        yte,
        best_predictions,
        target_names=[str(digit) for digit in range(10)],
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(yte, best_predictions, labels=list(range(10)))
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(cm, cmap="Blues")
ax.set_title(f"Confusion matrix: {best_row['Optimizer']}")
ax.set_xlabel("Predicted digit")
ax.set_ylabel("True digit")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticklabels(range(10))
for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        color = "white" if cm[row, column] > cm.max() / 2 else "black"
        ax.text(column, row, str(cm[row, column]), ha="center", va="center", fontsize=8, color=color)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp09_best_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Inference with the best optimizer

In [ ]:
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(Xte), size=10, replace=False)
sample_probabilities = best_model.predict(Xte[sample_indices], verbose=0)
sample_predictions = sample_probabilities.argmax(axis=1)
sample_confidences = sample_probabilities.max(axis=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, index, prediction, confidence in zip(
    axes.ravel(), sample_indices, sample_predictions, sample_confidences
):
    ax.imshow(Xte[index].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    true_label = int(yte[index])
    color = "green" if prediction == true_label else "red"
    ax.set_title(
        f"true {true_label} / pred {int(prediction)}\nconf {confidence:.3f}",
        fontsize=8,
        color=color,
    )
    ax.axis("off")
fig.suptitle(f"Inference with {best_row['Optimizer']}", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp09_sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()


def predict_digit(image):
    image = np.asarray(image, dtype=np.float32)
    if image.shape == (28, 28):
        image = image.reshape(1, 784)
    if image.max() > 1.0:
        image = image / 255.0
    probabilities = best_model.predict(image, verbose=0)[0]
    digit = int(np.argmax(probabilities))
    return digit, float(probabilities[digit]), probabilities


predicted_digit, confidence, all_probabilities = predict_digit(X_test_raw[0])
print(f"Single-digit inference: true={int(y_test_raw[0])}, "
      f"pred={predicted_digit}, confidence={confidence:.4f}")
print("Probabilities:", np.round(all_probabilities, 4).tolist())

## 9. Save the best model

In [ ]:
best_model_path = OUTPUT_DIR / "exp09_best_mnist_optimizer.keras"
best_model.save(best_model_path)
print(f"Saved best model to: {best_model_path.resolve()}")

reloaded = tf.keras.models.load_model(best_model_path)
reloaded_predictions = reloaded.predict(Xte, verbose=0).argmax(axis=1)
assert np.array_equal(reloaded_predictions, best_predictions)
print("Reload check passed: predictions are identical.")

## 10. Conclusion

All six optimizers train the same dense MNIST classifier, but they differ in
convergence speed and final accuracy. In the original lab record, SGD (97.42%),
Momentum (97.34%), Nesterov (97.23%), and Adam (97.05%) were close, while
Mini-Batch SGD (94.46%) and Adagrad (90.50%) lagged. The ordering can change
with the number of epochs and the random seed, but the experiment shows that
optimizer choice materially affects training.

**Artifacts created at runtime (not committed):**

- `outputs/exp09_best_mnist_optimizer.keras`
- `outputs/exp09_optimizer_results.csv`
- `outputs/exp09_validation_curves.png`
- `outputs/exp09_accuracy_and_time.png`
- `outputs/exp09_best_confusion_matrix.png`
- `outputs/exp09_sample_predictions.png`

**Execution note:** the notebook was syntax-checked during repository
preparation; the full six-optimizer training run was not performed locally.
Set `FAST_DEV_RUN = True` for a quick functional check.